# XAU preprocessing example

This notebook runs the `preprocess` class on the repository's XAU 15-minute candles. It produces cleaned indicator/target data, candle-quality reports, and weekly walk-forward folds. Audit flags are for review and are not model features.

In [ ]:
from pathlib import Path
import sys

working_dir = Path.cwd().resolve()
repo_root = next(
    (path for path in (working_dir, *working_dir.parents)
     if (path / "data_prep" / "preprocess.py").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Could not locate the repository containing data_prep/preprocess.py")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from data_prep.preprocess import preprocess

In [ ]:
processor = preprocess(
    symbols=["XAU"],
    timeframes=["15m"],
    write_outputs=True,
    train_size=12,
    val_size=2,
    test_size=2,
    step=2,
)

source_file = processor.data_dir / "XAU" / "XAU_15min.csv"
print(f"Input:  {source_file} (exists: {source_file.is_file()})")
print(f"Output: {processor.output_dir / 'XAU' / 'XAU_15m_features.csv'}")
if not source_file.is_file():
    raise FileNotFoundError(source_file)

## Process candles and inspect quality

The run writes the enriched CSV under `datapoints_preprocessed/`. It returns a processing summary, feature warm-up audit, and a sample of the newest rows.

In [ ]:
quality_summary, feature_audit, preview = processor.run_preprocessing()
display(quality_summary)

features_missing_after_warmup = feature_audit.loc[
    feature_audit["missing_after_warmup"] > 0
]
display(features_missing_after_warmup.head(30))
display(preview)

In [ ]:
source_quality, gap_events, price_events = processor.get_quality_reports("XAU", "15m")
print(f"Gap events: {len(gap_events)}; price events: {len(price_events)}")
display(source_quality)
display(gap_events.head(10))
display(price_events.head(10))

## Select a walk-forward fold

Fold sizes count observed seven-day bins. `select_fold` materializes the requested fold from the saved processed CSV, and excludes next-bar labels that would cross the train/validation/test boundary.

In [ ]:
fold_summary = processor.get_fold_summary("XAU", "15m")
display(fold_summary.head())
display(fold_summary.tail())

train_data, validation_data, test_data = processor.select_fold(
    0, symbol="XAU", timeframe="15m"
)
for name, frame in (("train", train_data), ("validation", validation_data), ("test", test_data)):
    print(f"{name}: {len(frame):,} rows, {frame.time.min()} to {frame.time.max()}")

display(train_data[["time", "close", *processor.FEATURE_COLUMNS, "target_end", "target_up"]].tail())